
# Домашнее задание 4.3. Бассейны Ньютона при слиянии корней

## 1. Математическая модель

Рассматривается семейство полиномов

$$
f_\varepsilon(z)=(z+1)(z-1)(z-\varepsilon),
\qquad z\in\mathbb C,
$$

где $\varepsilon$ — вещественный параметр.

Для поиска корней используется метод Ньютона:

$$
z_{n+1}=z_n-\frac{f_\varepsilon(z_n)}
{f_\varepsilon'(z_n)}.
$$

Производная полинома:

$$
f_\varepsilon'(z)=3z^2-2\varepsilon z-1.
$$

При $\varepsilon=0$ полином имеет три различных корня:

$$
z_1=-1,\qquad z_2=0,\qquad z_3=1.
$$

При $\varepsilon\to1$ два корня сближаются. При $\varepsilon=1$ получаем

$$
f_1(z)=(z+1)(z-1)^2,
$$

то есть $z=1$ становится двойным корнем.

### Бассейны притяжения

Бассейном притяжения корня $z_k$ называется множество начальных точек $z_0$, для которых последовательность метода Ньютона сходится к $z_k$.

На первом этапе используем:

- область $\operatorname{Re}z,\operatorname{Im}z\in[-2,2]$;
- сетку $600\times600$;
- максимальное число итераций — 100;
- параметр $\varepsilon=0$.

Сходимость проверяется одновременно по расстоянию до известного корня и по невязке:

$$
|z_n-z_k|\le\tau_{\mathrm{root}},
\qquad
|f_\varepsilon(z_n)|\le\tau_{\mathrm{res}}.
$$

При поиске корней отдельно различаем:

1. Успешную сходимость к одному из корней.
2. Численный сбой (например, деление на почти нулевую производную).
3. Отсутствие установленной сходимости за заданное число итераций.

Проверка сходимости выполняется **до вычисления очередного шага Ньютона**.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# Область вычислений
DOMAIN = (-2.0, 2.0, -2.0, 2.0)
GRID_SIZE = 600
MAX_ITER = 100

# Допуски
TOL_ROOT = 1e-8
TOL_RES = 1e-10
TOL_DERIV = 1e-14

def f(z, eps):
    return (z + 1) * (z - 1) * (z - eps)

def df(z, eps):
    return (
        (z - 1) * (z - eps)
        + (z + 1) * (z - eps)
        + (z + 1) * (z - 1)
    )

def get_roots(eps):
    # Метки: 1 = -1, 2 = +1, 3 = eps
    # При eps=1 двойной корень имеет одну метку
    if eps == 1.0:
        roots = np.array([-1 + 0j, 1 + 0j])
        ids = np.array([1, 2], dtype=np.int8)
    else:
        roots = np.array([-1 + 0j, 1 + 0j, complex(eps)])
        ids = np.array([1, 2, 3], dtype=np.int8)

    return roots, ids

def root_tolerance(eps, tol_root=TOL_ROOT):
    roots, _ = get_roots(eps)

    # Минимальное расстояние между различными корнями
    distances = np.abs(roots[:, None] - roots[None, :])
    np.fill_diagonal(distances, np.inf)

    min_distance = np.min(distances)

    # Допуск существенно меньше расстояния между корнями
    return min(tol_root, min_distance / 1000)


print("Корни при eps=0:", get_roots(0)[0])
print("Корни при eps=1:", get_roots(1)[0])
print()
print("Допуск по корню:", TOL_ROOT)
print("Допуск по невязке:", TOL_RES)
print("Порог для производной:", TOL_DERIV)


In [ ]:

def newton_basins(
    eps,
    grid_size=GRID_SIZE,
    max_iter=MAX_ITER,
    tol_root=TOL_ROOT,
    tol_res=TOL_RES,
    tol_deriv=TOL_DERIV,
    domain=DOMAIN
):
    xmin, xmax, ymin, ymax = domain

    real = np.linspace(xmin, xmax, grid_size)
    imag = np.linspace(ymin, ymax, grid_size)

    X, Y = np.meshgrid(real, imag)
    z = (X + 1j * Y).ravel().copy()

    # 0 — сходимость пока не установлена
    labels = np.zeros(z.size, dtype=np.int8)

    # Число выполненных шагов Ньютона
    iterations = np.full(
        z.size, max_iter, dtype=np.int32
    )

    roots, root_ids = get_roots(eps)
    actual_tol = root_tolerance(eps, tol_root)

    for k in range(max_iter + 1):
        active = np.flatnonzero(labels == 0)

        if active.size == 0:
            break

        za = z[active]

        # 1. Проверка сходимости ДО деления
        with np.errstate(over="ignore", invalid="ignore"):
            residual = np.abs(f(za, eps))

            distances = np.abs(
                za[:, None] - roots[None, :]
            )

            nearest = np.argmin(distances, axis=1)

            nearest_distance = distances[
                np.arange(active.size), nearest
            ]

            converged = (
                (nearest_distance <= actual_tol)
                & (residual <= tol_res)
            )

        # Записываем найденные корни
        good_indices = active[converged]

        labels[good_indices] = root_ids[nearest[converged]]
        iterations[good_indices] = k

        # На последней итерации только проверяем сходимость
        if k == max_iter:
            break

        # 2. Продолжаем только для несошедшихся точек
        remaining = active[~converged]
        zr = z[remaining]

        with np.errstate(
            divide="ignore",
            invalid="ignore",
            over="ignore"
        ):
            numerator = f(zr, eps)
            denominator = df(zr, eps)

            # Проверка численной корректности
            failed = (
                ~np.isfinite(numerator)
                | ~np.isfinite(denominator)
                | (np.abs(denominator) < tol_deriv)
            )

            z_new = np.full(
                zr.shape, np.nan + 1j * np.nan
            )

            safe = ~failed

            z_new[safe] = (
                zr[safe]
                - numerator[safe] / denominator[safe]
            )

            failed |= ~np.isfinite(z_new)

        # Численные сбои
        failed_indices = remaining[failed]
        labels[failed_indices] = -1
        iterations[failed_indices] = k

        # Успешно выполненный шаг
        valid_indices = remaining[~failed]
        z[valid_indices] = z_new[~failed]

    return (
        labels.reshape(grid_size, grid_size),
        iterations.reshape(grid_size, grid_size)
    )


# Основной расчёт
labels_0, iterations_0 = newton_basins(eps=0.0)

print("Результаты при eps=0:")
print(f"Корень -1: {np.sum(labels_0 == 1)}")
print(f"Корень +1: {np.sum(labels_0 == 2)}")
print(f"Корень  0: {np.sum(labels_0 == 3)}")
print(f"Численные сбои: {np.sum(labels_0 == -1)}")
print(f"Не определено: {np.sum(labels_0 == 0)}")


In [ ]:

# Цвета для меток -1, 0, 1, 2, 3
BASIN_COLORS = [
    "#343434",  # -1: численный сбой
    "#F5F5F5",  #  0: не определено
    "#3478B8",  #  1: корень -1
    "#E79B39",  #  2: корень +1
    "#8861B1"   #  3: корень eps
]

BASIN_CMAP = ListedColormap(BASIN_COLORS)
BASIN_NORM = BoundaryNorm(
    np.arange(-1.5, 4.0, 1.0),
    BASIN_CMAP.N
)

def basin_legend(eps):
    labels = [
        "Численный сбой",
        "Не определено за лимит",
        "Корень -1",
        "Корень +1 (двойной)" if eps == 1.0 else "Корень +1"
    ]

    patches = [
        Patch(color=color, label=label)
        for color, label in zip(BASIN_COLORS[:4], labels)
    ]

    if eps != 1.0:
        patches.append(
            Patch(color=BASIN_COLORS[4], label="Корень eps")
        )

    return patches


def plot_basin(ax, labels, eps, title=None):
    ax.imshow(
        labels,
        origin="lower",
        extent=DOMAIN,
        cmap=BASIN_CMAP,
        norm=BASIN_NORM,
        interpolation="nearest"
    )

    roots, _ = get_roots(eps)

    ax.scatter(
        roots.real,
        roots.imag,
        marker="*",
        s=100,
        c="white",
        edgecolors="black",
        linewidths=1.2,
        zorder=5
    )

    ax.set_xlabel("Re(z)")
    ax.set_ylabel("Im(z)")
    ax.set_aspect("equal")

    if title is not None:
        ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Карта бассейнов
plot_basin(
    axes[0],
    labels_0,
    eps=0.0,
    title="Бассейны Ньютона, eps = 0"
)

# Карта числа итераций
# Неуспешные точки оставляем в цветах их категорий
axes[1].imshow(
    labels_0,
    origin="lower",
    extent=DOMAIN,
    cmap=BASIN_CMAP,
    norm=BASIN_NORM,
    interpolation="nearest"
)

iteration_map = np.ma.masked_where(
    labels_0 <= 0,
    iterations_0
)

iteration_cmap = plt.colormaps["magma"].copy()
iteration_cmap.set_bad((0, 0, 0, 0))

im = axes[1].imshow(
    iteration_map,
    origin="lower",
    extent=DOMAIN,
    cmap=iteration_cmap,
    vmin=0,
    vmax=MAX_ITER,
    interpolation="nearest"
)

roots, _ = get_roots(0.0)

axes[1].scatter(
    roots.real,
    roots.imag,
    marker="*",
    s=100,
    c="white",
    edgecolors="black",
    linewidths=1.2
)

axes[1].set_title("Число итераций до сходимости")
axes[1].set_xlabel("Re(z)")
axes[1].set_ylabel("Im(z)")
axes[1].set_aspect("equal")

fig.colorbar(
    im,
    ax=axes[1],
    label="Число итераций",
    shrink=0.82
)

fig.legend(
    handles=basin_legend(0.0),
    loc="lower center",
    ncol=5,
    fontsize=9
)

plt.tight_layout(rect=(0, 0.10, 1, 1))
plt.show()


In [ ]:

iteration_limits = [5, 20, 100, 200]

print(
    f"{'Лимит':>8} "
    f"{'Сошлось':>12} "
    f"{'Не определено':>16} "
    f"{'Сбои':>10}"
)
print("-" * 52)

for limit in iteration_limits:
    labels_test, _ = newton_basins(
        eps=0.0,
        max_iter=limit
    )

    converged = np.sum(labels_test > 0)
    unresolved = np.sum(labels_test == 0)
    failed = np.sum(labels_test == -1)

    print(
        f"{limit:>8} "
        f"{converged:>12} "
        f"{unresolved:>16} "
        f"{failed:>10}"
    )


In [ ]:

def newton_trace(
    z0,
    eps,
    max_iter=MAX_ITER,
    tol_root=TOL_ROOT,
    tol_res=TOL_RES,
    tol_deriv=TOL_DERIV
):
    roots, ids = get_roots(eps)
    actual_tol = root_tolerance(eps, tol_root)

    z = complex(z0)
    trajectory = [z]

    for k in range(max_iter + 1):
        nearest = np.argmin(np.abs(z - roots))

        # Проверка корня до вычисления следующего шага
        if (
            abs(z - roots[nearest]) <= actual_tol
            and abs(f(z, eps)) <= tol_res
        ):
            return np.array(trajectory), int(ids[nearest])

        if k == max_iter:
            return np.array(trajectory), 0

        derivative = df(z, eps)
        residual = f(z, eps)

        if (
            not np.isfinite(derivative)
            or not np.isfinite(residual)
            or abs(derivative) < tol_deriv
        ):
            return np.array(trajectory), -1

        with np.errstate(
            divide="ignore",
            over="ignore",
            invalid="ignore"
        ):
            z_new = z - residual / derivative

        if not np.isfinite(z_new):
            return np.array(trajectory), -1

        z = z_new
        trajectory.append(z)

    return np.array(trajectory), 0


examples = [
    ("К корню -1", -1.4 + 0.15j, 100),
    ("К корню 0", -0.2 + 0.13j, 100),
    ("К корню +1", 0.7 + 0.1j, 100),
    ("Критическая точка", 1 / np.sqrt(3), 100),
    ("Малый лимит", 0.7 + 0.1j, 6)
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for ax, (title, z0, limit) in zip(axes, examples):
    trajectory, status = newton_trace(
        z0, eps=0.0, max_iter=limit
    )

    ax.plot(
        trajectory.real,
        trajectory.imag,
        "-o",
        markersize=4,
        linewidth=1.2
    )

    # Нумерация приближений
    for n, point in enumerate(trajectory):
        ax.annotate(
            str(n),
            (point.real, point.imag),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8
        )

    roots, _ = get_roots(0.0)
    ax.scatter(
        roots.real,
        roots.imag,
        marker="*",
        s=120,
        c="gold",
        edgecolors="black",
        zorder=5
    )

    status_text = {
        -1: "Численный сбой",
         0: "Не определено",
         1: "Корень -1",
         2: "Корень +1",
         3: "Корень 0"
    }[status]

    ax.set_title(f"{title}\n{status_text}")
    ax.set_xlabel("Re(z)")
    ax.set_ylabel("Im(z)")
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

    print(
        f"{title}: {status_text}, "
        f"число шагов = {len(trajectory)-1}"
    )

axes[-1].axis("off")

plt.tight_layout()
plt.show()



## 2. Результаты при $\varepsilon=0$

Для полинома

$$
f_0(z)=z(z^2-1)
$$

построены бассейны притяжения трёх корней $-1$, $0$, $1$ на сетке $600\times600$.

Границы бассейнов имеют сложную структуру: близкие начальные точки могут сходиться к разным корням.

Карта числа итераций показывает, что скорость сходимости зависит от начального приближения. Вблизи границ бассейнов метод Ньютона может требовать значительно больше шагов.

Для численной классификации использованы допуски

$$
\tau_{\mathrm{root}}=10^{-8},\qquad
\tau_{\mathrm{res}}=10^{-10}
$$

и порог для производной

$$
\tau_{\mathrm{deriv}}=10^{-14}.
$$

Сходимость проверялась одновременно по расстоянию до известного корня и по невязке, до деления на производную.

Численные сбои и отсутствие установленной сходимости за заданное число шагов учитывались как отдельные категории.

### Можно ли считать точку вне бассейнов, если метод не сошёлся за установленное число итераций?

Нет. Отсутствие установленной сходимости при конечном лимите не доказывает, что начальная точка не принадлежит бассейну.

При увеличении числа итераций такие точки могут сходиться к одному из корней.

Кроме того, численные сбои необходимо отличать от медленной сходимости: например, при малой производной вычисление шага Ньютона становится чувствительным к ошибкам округления.

Следовательно, классификация бассейнов при конечном числе итераций является численным приближением, зависящим от заданных допусков и лимита.



## 3. Бассейны Ньютона при сближении корней

Рассмотрим четыре значения параметра:

$$
\varepsilon\in\{0.5,\;0.9,\;0.99,\;1\}.
$$

При $\varepsilon<1$ корни $z=1$ и $z=\varepsilon$ различны, но расстояние между ними

$$
d=1-\varepsilon
$$

стремится к нулю.

Чтобы не перепутать близкие корни при классификации, используем условие

$$
\tau_{\mathrm{root}}
=
\min\left(10^{-8},\frac{d_{\min}}{1000}\right),
$$

где $d_{\min}$ — минимальное расстояние между различными корнями.

При $\varepsilon=1$ корни $z=1$ и $z=\varepsilon$ совпадают. Двойной корень учитывается как один исход.

При этом метод Ньютона остаётся стандартным, без поправки на кратность корня.


In [ ]:

eps_values = [0.5, 0.9, 0.99, 1.0]

basin_results = {}

fig, axes = plt.subplots(2, 2, figsize=(12, 11))

for ax, eps in zip(axes.ravel(), eps_values):
    labels, steps = newton_basins(
        eps=eps,
        grid_size=600,
        max_iter=100
    )

    basin_results[eps] = (labels, steps)

    plot_basin(
        ax,
        labels,
        eps,
        title=f"eps = {eps}"
    )

    print(
        f"eps={eps:.2f}, "
        f"tol_root={root_tolerance(eps):.2e}, "
        f"не определено={np.sum(labels == 0)}, "
        f"сбои={np.sum(labels == -1)}"
    )

fig.suptitle(
    "Бассейны Ньютона при сближении корней",
    fontsize=15
)

fig.legend(
    handles=basin_legend(0.99),
    loc="lower center",
    ncol=5,
    fontsize=9
)

plt.tight_layout(rect=(0, 0.06, 1, 0.97))
plt.show()


In [ ]:

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Параметр движется от 0.5 к 1.
# Больше кадров около точки слияния корней.
eps_frames = np.concatenate([
    np.linspace(0.5, 0.9, 8, endpoint=False),
    np.linspace(0.9, 0.99, 8, endpoint=False),
    np.linspace(0.99, 1.0, 9)
])

# Предварительно вычисляем карты
animation_maps = []

for eps in eps_frames:
    labels, _ = newton_basins(
        eps=float(eps),
        grid_size=240,
        max_iter=100
    )
    animation_maps.append(labels)

fig, ax = plt.subplots(figsize=(7, 6), dpi=80)

im = ax.imshow(
    animation_maps[0],
    origin="lower",
    extent=DOMAIN,
    cmap=BASIN_CMAP,
    norm=BASIN_NORM,
    interpolation="nearest"
)

roots, _ = get_roots(float(eps_frames[0]))

moving_roots, = ax.plot(
    roots.real,
    roots.imag,
    linestyle="none",
    marker="*",
    markersize=12,
    markerfacecolor="white",
    markeredgecolor="black"
)

ax.set_xlabel("Re(z)")
ax.set_ylabel("Im(z)")
ax.set_aspect("equal")


def update_basins(frame):
    eps = float(eps_frames[frame])

    im.set_data(animation_maps[frame])

    roots, _ = get_roots(eps)

    moving_roots.set_data(
        roots.real,
        roots.imag
    )

    ax.set_title(
        f"Бассейны Ньютона, eps = {eps:.5f}"
    )

    return im, moving_roots


anim_basins = FuncAnimation(
    fig,
    update_basins,
    frames=len(eps_frames),
    interval=125,
    blit=False,
    repeat=True
)

plt.close(fig)

# Увеличиваем лимит размера встроенной анимации
plt.rcParams["animation.embed_limit"] = 50

display(
    HTML(anim_basins.to_jshtml(fps=8, default_mode="loop"))
)


In [ ]:

# Исследуем сходимость при eps=1
eps = 1.0

test_points = [
    ("Простой корень -1", -1.6 + 0.5j, -1.0),
    ("Двойной корень +1", 1.3 + 0.2j, 1.0)
]

convergence_data = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, z0, exact_root) in zip(axes, test_points):
    trajectory, status = newton_trace(
        z0,
        eps=eps,
        max_iter=100,
        tol_root=1e-12,
        tol_res=1e-14
    )

    n = np.arange(len(trajectory))

    # Ошибка относительно известного точного корня
    errors = np.abs(trajectory - exact_root)

    # Невязка
    residuals = np.abs(f(trajectory, eps))

    convergence_data[name] = {
        "n": n,
        "errors": errors,
        "residuals": residuals,
        "status": status
    }

    # Отбрасываем точные нули для логарифмического графика
    valid_e = errors > 0
    valid_r = residuals > 0

    ax.semilogy(
        n[valid_e],
        errors[valid_e],
        "-o",
        markersize=4,
        label="Ошибка |z_n - z*|"
    )

    ax.semilogy(
        n[valid_r],
        residuals[valid_r],
        "-s",
        markersize=3,
        label="Невязка |f(z_n)|"
    )

    ax.set_xlabel("Номер итерации")
    ax.set_ylabel("Абсолютная величина")
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3, which="both")

    print(f"\n{name}")
    print(f"Статус: {status}")
    print(f"Число шагов: {len(trajectory)-1}")
    print(f"Конечная ошибка: {errors[-1]:.6e}")
    print(f"Конечная невязка: {residuals[-1]:.6e}")

fig.suptitle(
    "Сходимость метода Ньютона при eps = 1",
    fontsize=14
)

plt.tight_layout()
plt.show()


In [ ]:

def observed_orders(
    errors,
    min_error=1e-11,
    max_error=0.1
):
    e0 = errors[:-2]
    e1 = errors[1:-1]
    e2 = errors[2:]

    # Оставляем участок устойчивого убывания ошибки,
    # не слишком далеко от корня, но и не на уровне
    # машинного округления / критерия остановки.
    valid = (
        (e0 < max_error)
        & (e2 > min_error)
        & (e0 > e1)
        & (e1 > e2)
    )

    indices = np.arange(1, len(errors)-1)[valid]

    p = (
        np.log(e2[valid] / e1[valid])
        / np.log(e1[valid] / e0[valid])
    )

    return indices, p


fig, ax = plt.subplots(figsize=(10, 5))

for name, data in convergence_data.items():
    errors = data["errors"]

    indices, orders = observed_orders(errors)

    ax.plot(
        indices,
        orders,
        "-o",
        markersize=4,
        label=name
    )

    print(f"\n{name}")
    print("Использованные номера итераций:", indices)

    if len(orders) > 0:
        print(
            f"Медианный наблюдаемый порядок: "
            f"{np.median(orders):.5f}"
        )

    # Для двойного корня оцениваем коэффициент сжатия
    if "Двойной" in name:
        mask = (
            (errors[:-1] > 1e-6)
            & (errors[:-1] < 1e-2)
        )

        ratios = errors[1:][mask] / errors[:-1][mask]

        if len(ratios) > 0:
            print(
                f"Медианное отношение e_(n+1)/e_n: "
                f"{np.median(ratios):.6f}"
            )

ax.axhline(
    2,
    color="tab:blue",
    linestyle="--",
    alpha=0.6,
    label="Теоретический порядок p=2"
)

ax.axhline(
    1,
    color="tab:orange",
    linestyle="--",
    alpha=0.6,
    label="Теоретический порядок p=1"
)

ax.set_xlabel("Номер итерации n")
ax.set_ylabel("Наблюдаемый порядок p_n")
ax.set_title("Оценка порядка сходимости метода Ньютона")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

settings = [
    (20,  1e-8,  1e-10),
    (100, 1e-8,  1e-10),
    (200, 1e-8,  1e-10),
    (100, 1e-6,  1e-8),
    (100, 1e-10, 1e-12)
]

print(
    f"{'eps':>6} {'Лимит':>8} "
    f"{'tol_root':>12} {'tol_res':>12} "
    f"{'Сошлось, %':>13} {'Не опр.':>10} {'Сбои':>8}"
)
print("-" * 80)

for eps in [0.99, 1.0]:
    for max_iter, tol_root, tol_res in settings:
        labels, steps = newton_basins(
            eps=eps,
            grid_size=300,
            max_iter=max_iter,
            tol_root=tol_root,
            tol_res=tol_res
        )

        converged_pct = 100 * np.mean(labels > 0)
        unresolved = np.sum(labels == 0)
        failed = np.sum(labels == -1)

        print(
            f"{eps:>6.2f} {max_iter:>8} "
            f"{tol_root:>12.1e} {tol_res:>12.1e} "
            f"{converged_pct:>13.3f} "
            f"{unresolved:>10} {failed:>8}"
        )


In [ ]:

test_cases = [
    (0.99, 1.0 + 1e-4),
    (0.999999, 0.9999995),
    (1.0, 1.0 + 1e-4),
    (1.0, 1.0 + 1e-6)
]

print(
    f"{'eps':>12} "
    f"{'Расстояние до корня':>24} "
    f"{'Невязка':>16}"
)
print("-" * 57)

for eps, z in test_cases:
    roots, _ = get_roots(eps)

    error = np.min(np.abs(z - roots))
    residual = abs(f(z, eps))

    print(
        f"{eps:>12.6f} "
        f"{error:>24.8e} "
        f"{residual:>16.8e}"
    )



## 4. Выводы по жёлтому уровню

Исследовано влияние параметра $\varepsilon$ на бассейны притяжения метода Ньютона для полинома

$$
f_\varepsilon(z)=(z+1)(z-1)(z-\varepsilon).
$$

### Изменение бассейнов

Построены карты при $\varepsilon=0.5$, $0.9$, $0.99$ и $1$, а также анимация изменения бассейнов при непрерывном изменении параметра.

При сближении корней $z=\varepsilon$ и $z=1$ изменяется геометрия бассейнов притяжения.

При $\varepsilon=1$ корни совпадают, образуя двойной корень $z=1$, который учитывается как один исход.

### Порядок сходимости

Для простого корня $z=-1$ при $\varepsilon=1$ наблюдается квадратичная сходимость:

$$
e_{n+1}\sim Ce_n^2,\qquad p\approx2.
$$

Для двойного корня $z=1$ стандартный метод Ньютона имеет линейную сходимость:

$$
\frac{e_{n+1}}{e_n}\to\frac12,\qquad p\approx1.
$$

Порядок сходимости оценён непосредственно по последовательности ошибок относительно точных корней, с исключением начального участка и области, где существенно влияние округления и критерия остановки.

### Влияние допусков и числа итераций

Численная классификация бассейнов зависит от допусков и максимального числа шагов.

При двойном корне увеличение лимита особенно важно из-за линейной, а не квадратичной сходимости.

Отсутствие установленной сходимости за конечное число итераций не означает, что точка не принадлежит бассейну притяжения.

### Связь невязки и ошибки

При сближении простых корней производная в корнях уменьшается. Поэтому отношение ошибки к невязке может становиться большим.

Для двойного корня $z=1$:

$$
|f_1(z)|\approx2|z-1|^2.
$$

Следовательно, очень малая невязка может соответствовать существенно большей ошибке.

В общем случае для рассматриваемого монического полинома справедлива оценка

$$
\min_k|z-z_k|\le |f_\varepsilon(z)|^{1/3}.
$$

Однако она достаточно груба и не заменяет проверку близости к известному корню.

Таким образом, для надёжной классификации необходимо учитывать расстояния до корней, невязку, кратность корней, численные допуски и лимит итераций.
